# Audiobook AI - Colab GPU Runner

Notebook này chạy đúng kiến trúc mới:

`Streamlit UI -> FastAPI API -> SQLite Queue -> GPU XTTS Pipeline -> Download audio`

Colab sẽ host cả FastAPI và Streamlit, rồi mở 2 Cloudflare quick tunnels.

## 0. Bật GPU

Trong Colab chọn `Runtime -> Change runtime type -> T4 GPU` trước khi chạy notebook.

Lưu ý: Coqui TTS/XTTS không ổn định trên Python 3.12. Nếu Colab runtime của bạn là Python 3.12 và cell cài `TTS==0.22.0` lỗi, cần dùng runtime Python 3.10/3.11 hoặc chạy bằng Docker/máy GPU riêng.

In [ ]:
# [1] Kiểm tra Python trước khi cài XTTS
import os, sys

print(sys.version)
if sys.version_info >= (3, 12):
    print('WARNING: Coqui TTS/XTTS thường không cài được trên Python >= 3.12.')
    print('Nếu pip install TTS lỗi, hãy dùng Colab/runtime Python 3.10-3.11 hoặc máy GPU riêng.')

# Clone repo
REPO = "CSC15012-Final-Project-Audiobook-Generation-System"
GIT_URL = f"https://github.com/TiiAyyLuvBear/{REPO}"
BRANCH = "backend_implementation"

if not os.path.exists(REPO):
    !git clone --branch {BRANCH} {GIT_URL}
%cd {REPO}
!git fetch origin {BRANCH}
!git checkout {BRANCH}
!git pull origin {BRANCH}
print(os.getcwd())

In [ ]:
# [2] Kiểm tra GPU
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
print("Torch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# [3] Cài dependencies
# Coqui TTS khá nhạy version Python/package. Nếu cell này lỗi, restart runtime rồi chạy lại từ đầu.
!apt-get update -qq
!apt-get install -y -qq ffmpeg
!pip install -q -r requirements.txt
!pip install -q TTS==0.22.0
!pip install -q python-multipart
!pip install -q huggingface_hub

# Smoke import
import fastapi, streamlit
from TTS.api import TTS
print("Dependencies OK")

In [ ]:
# [4] Chuẩn bị .env cho Colab
from pathlib import Path

from huggingface_hub import snapshot_download

XTTS_HF_REPO = "aiMy144/XTTSv2VietAudiobook"
XTTS_LOCAL_DIR = "model"
HF_TOKEN = ""  # optional, chỉ cần nếu repo Hugging Face private

# Giống models/XTTSv2.ipynb: tải model từ Hugging Face về folder local
# rồi để hệ thống load model.pth/config.json/vocab.json từ folder đó.
snapshot_download(
    repo_id=XTTS_HF_REPO,
    repo_type="model",
    local_dir=XTTS_LOCAL_DIR,
    token=HF_TOKEN or None,
)
XTTS_MODEL_NAME_OR_PATH = XTTS_LOCAL_DIR

env_text = f"""API_BASE_URL=http://localhost:8000
CORS_ORIGINS=http://localhost:8501,http://127.0.0.1:8501
STORAGE_DIR=./storage
MAX_UPLOAD_MB=200
MAX_CONCURRENT_JOBS=1
TTS_ENGINE=xtts_gpu
XTTS_MODEL_NAME_OR_PATH={XTTS_MODEL_NAME_OR_PATH}
XTTS_CONFIG_PATH=
XTTS_VOCAB_PATH=
XTTS_VOICE_DIR=data/voice_samples
HF_TOKEN={HF_TOKEN}
"""
Path('.env').write_text(env_text, encoding='utf-8')
Path('storage').mkdir(exist_ok=True)
print(Path('.env').read_text())

## 5. Reference voices

XTTS cần file WAV trong `data/voice_samples`. Repo đã có vài file mẫu. Nếu bạn muốn clone giọng khác, upload `.wav` vào folder này và dùng `voice_id` trùng tên file.

In [ ]:
# [5] Kiểm tra reference voices
from pathlib import Path

voice_dir = Path('data/voice_samples')
voices = sorted(p.name for p in voice_dir.glob('*.wav'))
print(f"Found {len(voices)} reference voices")
for name in voices:
    print('-', name)

assert voices, 'Thiếu reference voice WAV trong data/voice_samples'

In [ ]:
# [6] Cài cloudflared
!wget -qO /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# [7] Chạy FastAPI + API tunnel
import os, re, socket, subprocess, sys, threading, time

API_PORT = 8000
api_log = []
api_url = [None]

subprocess.run(f'fuser -k {API_PORT}/tcp 2>/dev/null || true', shell=True)

def ready(port):
    try:
        socket.create_connection(('127.0.0.1', port), timeout=1).close()
        return True
    except OSError:
        return False

def start_api():
    env = os.environ.copy()
    env['TTS_ENGINE'] = 'xtts_gpu'
    env['STORAGE_DIR'] = './storage'
    env['XTTS_MODEL_NAME_OR_PATH'] = XTTS_MODEL_NAME_OR_PATH
    env['XTTS_VOICE_DIR'] = 'data/voice_samples'
    if HF_TOKEN:
        env['HF_TOKEN'] = HF_TOKEN
    p = subprocess.Popen(
        [sys.executable, '-m', 'uvicorn', 'src.app:app', '--host', '0.0.0.0', '--port', str(API_PORT)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )
    for line in p.stdout:
        api_log.append(line.rstrip())

threading.Thread(target=start_api, daemon=True).start()

print('Waiting for FastAPI', end='')
for _ in range(120):
    time.sleep(1)
    print('.', end='', flush=True)
    if ready(API_PORT):
        break
print()
assert ready(API_PORT), 'FastAPI chưa lên. Chạy cell xem api_log bên dưới.'
print('FastAPI local OK')

def start_api_tunnel():
    p = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://localhost:{API_PORT}'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in p.stdout:
        api_log.append(line.rstrip())
        m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if m:
            api_url[0] = m.group()
            break

threading.Thread(target=start_api_tunnel, daemon=True).start()

print('Waiting for API tunnel', end='')
for _ in range(60):
    time.sleep(1)
    print('.', end='', flush=True)
    if api_url[0]:
        break
print()
assert api_url[0], 'Không lấy được API tunnel URL'
print('API_PUBLIC_URL =', api_url[0])

In [ ]:
# [8] Chạy Streamlit + UI tunnel
import os, re, socket, subprocess, sys, threading, time

UI_PORT = 8501
ui_log = []
ui_url = [None]

subprocess.run(f'fuser -k {UI_PORT}/tcp 2>/dev/null || true', shell=True)

def start_streamlit():
    env = os.environ.copy()
    env['API_BASE_URL'] = api_url[0]
    p = subprocess.Popen(
        [sys.executable, '-m', 'streamlit', 'run', 'streamlit_app.py',
         f'--server.port={UI_PORT}', '--server.headless=true',
         '--server.enableCORS=false', '--server.enableXsrfProtection=false'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )
    for line in p.stdout:
        ui_log.append(line.rstrip())

threading.Thread(target=start_streamlit, daemon=True).start()

print('Waiting for Streamlit', end='')
for _ in range(60):
    time.sleep(1)
    print('.', end='', flush=True)
    if ready(UI_PORT):
        break
print()
assert ready(UI_PORT), 'Streamlit chưa lên. Chạy cell log bên dưới.'
print('Streamlit local OK')

def start_ui_tunnel():
    p = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://localhost:{UI_PORT}'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in p.stdout:
        ui_log.append(line.rstrip())
        m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if m:
            ui_url[0] = m.group()
            break

threading.Thread(target=start_ui_tunnel, daemon=True).start()

print('Waiting for UI tunnel', end='')
for _ in range(60):
    time.sleep(1)
    print('.', end='', flush=True)
    if ui_url[0]:
        break
print()
assert ui_url[0], 'Không lấy được UI tunnel URL'

print('=' * 80)
print('OPEN THIS STREAMLIT URL:', ui_url[0])
print('FastAPI URL:', api_url[0])
print('=' * 80)

In [ ]:
# [9] Xem logs khi cần debug
print('=== API LOG ===')
print('\n'.join(api_log[-200:]) or '(empty)')
print('\n=== STREAMLIT LOG ===')
print('\n'.join(ui_log[-200:]) or '(empty)')

## Cách test

1. Mở URL Streamlit ở cell `[8]`.
2. Upload EPUB.
3. Chọn `mp3` nếu `ffmpeg` đã cài OK, hoặc `wav` để debug nhanh hơn.
4. Bấm `Create audiobook job`.
5. Theo dõi progress/logs và download output khi completed.

Nếu job fail ngay ở TTS, kiểm tra:

- Colab có GPU chưa: `torch.cuda.is_available()` phải là `True`.
- `TTS==0.22.0` cài thành công chưa.
- Folder `data/voice_samples` có file WAV phù hợp chưa.